# ternary model comparison

In [ ]:
import numpy as np
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import imageio.v3 as iio
import os
from PIL import Image
import imagehash
import shutil
from sklearn.model_selection import train_test_split

import pandas as pd
from collections import Counter
import random
import seaborn as sns

Image_size = 96
Batch = 32
Random_seed = 42

# ported data processing

In [ ]:
def focal_loss_sparse(gamma, alpha):
    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        # One-hot encode y_true
        y_true_one_hot = tf.one_hot(y_true, depth=y_pred.shape[-1])
        
        # Cross entropy
        ce = -tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred), axis=-1)
        
        # Focal weight
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        
        return tf.reduce_mean(focal_weight * ce)
    return loss

def apply_blur(img):
    img = tf.expand_dims(img, 0)  
    img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
    return tf.squeeze(img, 0) 

def augment(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))    
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    image = tf.image.random_hue(image, max_delta=0.05)

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def compute_class_weights_Tri(ds):
    labels = np.concatenate([y.numpy() for _, y in ds.batch(512)])    
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    class_weights = dict(zip(classes, weights))
    print(f"Class weights: {class_weights}")
    return class_weights


def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [Image_size, Image_size])
    
    return image, label

In [ ]:
import tensorflow as tf
import numpy as np

# 1. Define your directory paths based on your folder structure
train_dir = 'Ternary_Training'
val_dir   = 'Ternary_Validation'
test_dir  = 'Ternary_Test'

# 2. Load Datasets from Folders
# Note: This automatically handles labels based on subfolder names
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode='int', 
    batch_size=Batch,
    image_size=(Image_size, Image_size),
    shuffle=True,
    seed=Random_seed
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    label_mode='int',
    batch_size=Batch,
    image_size=(Image_size, Image_size)
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    label_mode='int',
    batch_size=Batch,
    image_size=(Image_size, Image_size)
)


train_ds = (
    train_ds
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)


y_train = np.concatenate([y for x, y in train_ds], axis=0)
class_weights = compute_class_weights_Tri(y_train)

In [ ]:


model = keras.models.Sequential([
    
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
   
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.3),
   
    keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.GlobalAveragePooling2D(),
    
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(3, activation='softmax') 
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss= 'sparse_categorical_crossentropy',  
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    class_weight= class_weights,
)



In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

true_labels = []
y_pred_prob = []

print("Running model on test data...")
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    true_labels.extend(labels.numpy())
    y_pred_prob.extend(preds)

true_labels = np.array(true_labels)
y_pred_prob = np.array(y_pred_prob)

FIRE_THRESHOLD = 0.5

LAKE_THRESHOLD = 0.5

final_preds = []
# Replace your threshold loop with this:
final_preds = np.argmax(y_pred_prob, axis=1)
        
final_preds = np.array(final_preds)

print("\nAdjusted Classification Report:")
# Assuming your classes are 0: Fire, 1: Lake, 2: No_Fire based on your folder structure
print(classification_report(true_labels, final_preds, target_names=['Fire', 'Lake', 'No_Fire']))

print("\nAdjusted Confusion Matrix:")
print(confusion_matrix(true_labels, final_preds))


# og model to big. must add callbacks to make it better and shorten it. do with coparsion also makes false predictions often due to smoke or smt

In [ ]:
true_labels_arr = np.array(true_labels)
final_preds_arr = np.array(final_preds)

# Find No_Fire images predicted as Fire
no_fire_idx = class_to_idx['No_Fire']
fire_idx = class_to_idx['Fire']

misclassified = [
    test_p[i] for i in range(len(test_p))
    if true_labels_arr[i] == no_fire_idx and final_preds_arr[i] == fire_idx
]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, path in zip(axes.flat, misclassified[:10]):
    img = Image.open(path)
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('No_Fire misclassified as Fire')
plt.tight_layout()
plt.show()

# vary model sizs 

In [ ]:
def build_fire_model_scaled(model_size='medium', loss_fn=None):
    scales = {
        'very_small': (4, 8, 8),   
        'small':      (8, 16, 16),  
        'medium':     (8, 16, 16 ,32),  
        'large':      (8, 8, 16, 16, 32, 64),
    }
    num_blocks, base_filters, dense_dim = scales[model_size]
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(96, 96, 3)))

    for i in range(num_blocks):
        filters = base_filters * (2**i)
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.MaxPooling2D())
        model.add(keras.layers.Dropout(0.2 if i < 2 else 0.3))

    model.add(keras.layers.Conv2D(filters * 2, 3, activation='relu', padding='same'))
    model.add(keras.layers.GlobalAveragePooling2D())
    model.add(keras.layers.Dense(dense_dim, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(3, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=loss_fn,
        metrics=['accuracy'] # Removed precision/recall here to prevent Shape/Key Errors
    )
    return model

def run_experiment(size, loss_func, weights, callbacks):
    print(f"\n--- TESTING {size.upper()} ---")
    model = build_fire_model_scaled(size, loss_func)
    history = model.fit(train_ds, validation_data=val_ds, epochs=30, 
                        class_weight=weights, callbacks=callbacks, verbose=1)
    
    # Plotting only the metrics that exist
    metrics = ['loss', 'accuracy'] 
    plt.figure(figsize=(12, 5))
    for i, metric in enumerate(metrics):
        plt.subplot(1, 2, i+1)
        plt.plot(history.history[metric], label='Train')
        plt.plot(history.history[f'val_{metric}'], label='Val')
        plt.title(f'{size} {metric}')
        plt.legend()
    plt.show()
    return model, history

In [ ]:
# Configure your variables
focal_loss = 'sparse_categorical_crossentropy'
my_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

# Run the test
all_results = {}
for model_size in ['very_small', 'small', 'medium']:
    all_results[model_size] = run_experiment(model_size, focal_loss, class_weights, my_callbacks)

In [ ]:
def plot_model_comparison(results_dict):
    # 1. Extract final metrics from each history object
    data = []
    for size, history in results_dict.items():
        # Get the index of the best val_loss (best model)
        best_epoch = np.argmin(history.history['val_loss'])
        
        row = {
            'Size': size,
            'Val Accuracy': history.history['val_accuracy'][best_epoch],
            'Val Loss': history.history['val_loss'][best_epoch],
        }
        # Add precision/recall if they exist (for Binary)
        if 'val_precision' in history.history:
            row['Val Precision'] = history.history['val_precision'][best_epoch]
            row['Val Recall'] = history.history['val_recall'][best_epoch]
            
        data.append(row)

    df = pd.DataFrame(data)
    
    # Define the order for the X-axis
    order = ['very_small', 'small', 'medium', 'large', 'very_large']
    df['Size'] = pd.Categorical(df['Size'], categories=order, ordered=True)
    df = df.sort_values('Size')

    # 2. Create the "Pretty" Plot
    # We use a long-form format for Seaborn
    df_melted = df.melt(id_vars='Size', var_name='Metric', value_name='Value')
    
    # Separate Loss from other metrics for better scale
    df_metrics = df_melted[df_melted['Metric'] != 'Val Loss']
    df_loss = df_melted[df_melted['Metric'] == 'Val Loss']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    # Left Plot: Accuracy/Precision/Recall
    sns.barplot(data=df_metrics, x='Size', y='Value', hue='Metric', ax=ax1, palette='viridis')
    ax1.set_title('Performance Metrics Comparison', fontsize=15)
    ax1.set_ylim(0, 1.05)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # Right Plot: Loss (Lower is better)
    sns.lineplot(data=df_loss, x='Size', y='Value', marker='o', ax=ax2, color='red', linewidth=3)
    ax2.set_title('Validation Loss (Lower is Better)', fontsize=15)
    ax2.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

# To run it:
# plot_model_comparison(all_results)
def plot_model_comparison(results_dict):
    data = []
    for size, result in results_dict.items():
        # 1. Handle the tuple (model, history)
        # In your run_experiment, index 0 is model, index 1 is history
        if isinstance(result, tuple):
            history_obj = result[1] 
        else:
            history_obj = result
        
        # 2. Extract the actual dictionary of metrics
        h = history_obj.history 
        
        # 3. Use 'h' (the dictionary) to find the best epoch
        best_epoch = np.argmin(h['val_loss'])
        
        row = {
            'Size': size,
            'Val Accuracy': h['val_accuracy'][best_epoch],
            'Val Loss': h['val_loss'][best_epoch],
        }
        
        # 4. Check for optional metrics using the dict 'h'
        if 'val_precision' in h:
            row['Val Precision'] = h['val_precision'][best_epoch]
            row['Val Recall'] = h['val_recall'][best_epoch]
            
        data.append(row)

    # --- Rest of your plotting code stays the same ---
    df = pd.DataFrame(data)
    order = ['very_small', 'small', 'medium','EffNet']
    df['Size'] = pd.Categorical(df['Size'], categories=order, ordered=True)
    df = df.sort_values('Size')

    df_melted = df.melt(id_vars='Size', var_name='Metric', value_name='Value')
    df_metrics = df_melted[df_melted['Metric'] != 'Val Loss']
    df_loss = df_melted[df_melted['Metric'] == 'Val Loss']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    sns.barplot(data=df_metrics, x='Size', y='Value', hue='Metric', ax=ax1, palette='viridis')
    ax1.set_title('Performance Metrics Comparison', fontsize=15)
    ax1.set_ylim(0, 1.05)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    sns.lineplot(data=df_loss, x='Size', y='Value', marker='o', ax=ax2, color='red', linewidth=3)
    ax2.set_title('Validation Loss (Lower is Better)', fontsize=15)
    ax2.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()
# To run it:
plot_model_comparison(all_results)

# categorical vs focal loss

In [ ]:
losses_to_test = {
    'Sparse Categorical': 'sparse_categorical_crossentropy',
    'Categorical': 'categorical_crossentropy', 
    'Focal Sparse': focal_loss_sparse(gamma=2.0, alpha=0.25)
}

model_sizes = ['very_small', 'small', 'medium', 'large']
all_results = {}

# 2. Run the nested experiment
for loss_name, loss_fn in losses_to_test.items():
    print(f"--- Starting tests for Loss: {loss_name} ---")
    all_results[loss_name] = {}
    
    for model_size in model_sizes:
        print(f"Training {model_size} model...")
        # run_experiment should return the final validation accuracy or the history object
        result = run_experiment(model_size, loss_fn, class_weights, my_callbacks)
        
        # Extract the best validation accuracy (assuming result is a history object)
        # If your function returns a float, just use: all_results[loss_name][model_size] = result
        all_results[loss_name][model_size] = max(result.history['val_accuracy'])

# 3. Plotting the Comparison
plt.figure(figsize=(10, 6))

for loss_name, scores in all_results.items():
    # scores is a dict like {'very_small': 0.85, 'small': 0.88, ...}
    plt.plot(list(scores.keys()), list(scores.values()), marker='o', label=loss_name)

plt.title('Model Performance: Loss Function Comparison')
plt.xlabel('Model Size')
plt.ylabel('Best Validation Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Optional: Print results as a Table
df_results = pd.DataFrame(all_results)
print("\nFinal Comparison Table (Val Accuracy):")
print(df_results)
plot_model_comparison(df_results)

# closer comparison look

In [ ]:

def plot_model_analysiss(results_dict):
    # --- 1. DATA PROCESSING FOR BAR CHART ---
    data = []
    for size, result in results_dict.items():
        # Handle the (model, history) tuple from run_experiment
        history_obj = result[1] if isinstance(result, tuple) else result
        h = history_obj.history 
        
        # Find best epoch based on validation loss
        best_epoch = np.argmin(h['val_loss'])
        
        row = {
            'Size': size,
            'Accuracy': h['val_accuracy'][best_epoch],
        }
        # Add Precision/Recall if they exist in your compiled metrics
        if 'val_precision' in h: row['Precision'] = h['val_precision'][best_epoch]
        if 'val_recall' in h: row['Recall'] = h['val_recall'][best_epoch]
            
        data.append(row)

    df = pd.DataFrame(data)
    order = ['very_small', 'small', 'medium','EffNet']
    existing_order = [o for o in order if o in df['Size'].unique()]
    df['Size'] = pd.Categorical(df['Size'], categories=existing_order, ordered=True)
    df = df.sort_values('Size')

    # --- 2. SETUP PLOTS ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    colors = sns.color_palette("viridis", len(df))

    # --- LEFT: BAR PLOT (Accuracy/Precision/Recall) ---
    df_melted = df.melt(id_vars='Size', var_name='Metric', value_name='Value')
    sns.barplot(data=df_melted, x='Size', y='Value', hue='Metric', ax=ax1, palette='magma')
    ax1.set_title('Final Model Performance Comparison', fontsize=15, fontweight='bold')
    ax1.set_ylim(0, 1.1)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add values on top of bars
    for p in ax1.patches:
        ax1.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='baseline', fontsize=9, xytext=(0, 5), textcoords='offset points')

    # --- RIGHT: LINE PLOT (Loss Plateaus over Epochs) ---
    for (size, result), color in zip(results_dict.items(), colors):
        history_obj = result[1] if isinstance(result, tuple) else result
        h = history_obj.history
        epochs = range(1, len(h['val_loss']) + 1)
        
        # Plot Validation Loss for each model
        ax2.plot(epochs, h['val_loss'], label=f'{size}', linewidth=2.5, color=color, marker='o', markersize=4)

    ax2.set_title('Validation Loss Plateaus (Across All Models)', fontsize=15, fontweight='bold')
    ax2.set_xlabel('Epochs', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.legend(title='Model Size')
    ax2.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()
def plot_model_analysis(all_results):
    model_names = list(all_results.keys())  # Get all trained models
    num_models = len(model_names)
    
    # 1. Bar Chart for Final Accuracy
    plt.figure(figsize=(10, 6))
    final_accs = [all_results[m][1].history['val_accuracy'][-1] for m in model_names]
    
    plt.bar(model_names, final_accs, color=['skyblue', 'lightgreen', 'orange'][:num_models])
    plt.ylabel('Final Validation Accuracy')
    plt.title('Model Accuracy Comparison')
    for i, v in enumerate(final_accs):
        plt.text(i, v + 0.01, f"{v:.2f}", ha='center', fontweight='bold')
    plt.ylim(0, 1.1)
    plt.show()

    # 2. Training Progress Lines
    plt.figure(figsize=(12, 5))
    
    # Accuracy Plot
    plt.subplot(1, 2, 1)
    for name in model_names:
        history = all_results[name][1]
        plt.plot(history.history['val_accuracy'], label=f'{name} Val Acc')
    plt.title('Validation Accuracy Progress')
    plt.xlabel('Epochs')
    plt.legend()

    # Loss Plot
    plt.subplot(1, 2, 2)
    for name in model_names:
        history = all_results[name][1]
        plt.plot(history.history['val_loss'], label=f'{name} Val Loss')
    plt.title('Validation Loss Progress')
    plt.xlabel('Epochs')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
# Run the combined analysis
plot_model_analysis(all_results)

# useing efficient netV2

In [ ]:
def build_effnet_ternary_model(loss_fn=None):
    inputs = keras.layers.Input(shape=(96, 96, 3))
    
    # Scale [0, 1] back up to [0, 255] for EfficientNetV2's internal preprocessing
    x = keras.layers.Lambda(lambda tensor: tensor * 255.0)(inputs)

    # Load pre-trained EfficientNetV2B0
    base_model = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_tensor=x,
        input_shape=(96, 96, 3)
    )
    
    # Fine-tune the pre-trained weights
    base_model.trainable = True

    # Rebuild the top layers for 3 classes
    x = base_model.output
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(3, activation='softmax')(x)

    model = keras.models.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=loss_fn,
        metrics=['accuracy']
    )
    return model

# --- Run the EfficientNetV2 Experiment ---
print("\n--- TESTING EFFICIENTNETV2B0 (TERNARY) ---")
effnet_ter_model = build_effnet_ternary_model(focal_loss)

history_effnet_ter = effnet_ter_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=my_callbacks,
    verbose=1
)

# Append to your existing results dictionary
all_results['EfficientNetV2'] = (effnet_ter_model, history_effnet_ter)

# Now, re-run your plot function to see the comparison!
plot_model_analysis(all_results)

# compare again seperable and conv 

In [ ]:

# 2. Define the model builder function based on your 'small' scale (8 filters, 16 dense)
def model_builder(hp):
    # Fixed 'small' parameters from your notebook
    num_blocks = 8 
    base_filters = 16 
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(Image_size, Image_size, 3)))

    # Replicating your Conv2D block loop
    for i in range(num_blocks):
        filters = base_filters * (2**i)
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.Conv2D(filters, 3, activation='relu', padding='same'))
        model.add(keras.layers.MaxPooling2D())
        
        # Tune dropout per block layer
        hp_dropout = hp.Float(f'dropout_block_{i}', min_value=0.2, max_value=0.4, step=0.1)
        model.add(keras.layers.Dropout(hp_dropout))
        
        # Limit depth to prevent resource exhaustion during tuning
        if i >= 2: break 

    model.add(keras.layers.Conv2D(filters * 2, 3, activation='relu', padding='same'))
    model.add(keras.layers.GlobalAveragePooling2D())
    
    # Tune the dense dimension (your baseline was 16)
    hp_dense_units = hp.Int('dense_dim', min_value=16, max_value=64, step=16)
    model.add(keras.layers.Dense(hp_dense_units, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    
    # 3 classes for ternary output
    model.add(keras.layers.Dense(3, activation='softmax'))

    # Tune the learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy', # or focal_loss_sparse if you prefer
        metrics=['accuracy']
    )
    return model

# 3. Instantiate Hyperband
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory='hyperband_ternary_dir',
    project_name='fire_lake_tuning'
)

# 4. Run the search using your existing datasets and weights
stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[stop_early],
    class_weight=class_weights 
)

# 5. Retrieve best hyperparameters and build final model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Dense Units: {best_hps.get('dense_dim')}")
print(f"Best Learning Rate: {best_hps.get('learning_rate')}")

final_tuned_model = tuner.hypermodel.build(best_hps)
history = final_tuned_model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=30, 
    class_weight=class_weights,
    callbacks=my_callbacks
)